In [1]:
# Paso 1: importar las librerias necesarias para explorar los datos
 
import pandas as pd
import numpy as np
from pathlib import Path
 
# Ignorar warnings
# ==============================================================================
import warnings
warnings.filterwarnings("ignore")


# Configuración
# -----------------------------------------------------------------------
pd.set_option('display.max_columns', None) # para poder visualizar todas las columnas de los DataFrames

In [2]:
# Paso 2: definir rutas de entrada y salida

RUTA_INTERIM = Path("../data/interim")
RUTA_PROCESSED = Path("../data/processed")

RUTA_PROCESSED.mkdir(parents=True, exist_ok=True)

In [3]:
# Paso 3: cargar los datasets limpios generados en el notebook 02

def cargar_pkl(ruta):
    """
    Carga un archivo pickle.

    Parametros:
        ruta: ruta del archivo pickle.

    Devuelve:
        DataFrame con los datos cargados.
    """
    return pd.read_pickle(ruta)


customers = cargar_pkl(RUTA_INTERIM / "customers_clean.pkl")
orders = cargar_pkl(RUTA_INTERIM / "orders_clean.pkl")
items = cargar_pkl(RUTA_INTERIM / "items_clean.pkl")
payments = cargar_pkl(RUTA_INTERIM / "payments_clean.pkl")
products = cargar_pkl(RUTA_INTERIM / "products_clean.pkl")
categories = cargar_pkl(RUTA_INTERIM / "categories_clean.pkl")
holidays = cargar_pkl(RUTA_INTERIM / "holidays_clean.pkl")

In [4]:
holidays["holiday_date"].head(10)

596   2016-01-01
597   2016-02-09
598   2016-02-10
599   2016-03-25
600   2016-03-27
601   2016-04-21
602   2016-05-01
603   2016-05-26
604   2016-09-07
605   2016-10-12
Name: holiday_date, dtype: datetime64[ns]

In [5]:
# Paso 4: crear variables temporales a partir de la fecha de compra

def crear_variables_temporales(df, columna_fecha):
    """
    Crea variables temporales a partir de una columna de fecha.

    Parametros:
        df: DataFrame de entrada.
        columna_fecha: columna datetime base.

    Devuelve:
        DataFrame con nuevas variables temporales.
    """
    df = df.copy()

    df["order_purchase_date"] = df[columna_fecha].dt.normalize()
    df["purchase_year"] = df[columna_fecha].dt.year
    df["purchase_month"] = df[columna_fecha].dt.month
    df["purchase_day"] = df[columna_fecha].dt.day
    df["purchase_hour"] = df[columna_fecha].dt.hour
    df["purchase_dayofweek"] = df[columna_fecha].dt.dayofweek
    df["purchase_quarter"] = df[columna_fecha].dt.quarter

    return df


orders = crear_variables_temporales(
    orders,
    "order_purchase_timestamp"
)

orders[
    [
        "order_purchase_timestamp",
        "order_purchase_date",
        "purchase_year",
        "purchase_month",
        "purchase_hour",
        "purchase_dayofweek",
        "purchase_quarter"
    ]
].head()

,order_purchase_timestamp,order_purchase_date,purchase_year,purchase_month,purchase_hour,purchase_dayofweek,purchase_quarter
0,2017-10-02 10:56:33,2017-10-02,2017,10,10,0,4
1,2018-07-24 20:41:37,2018-07-24,2018,7,20,1,3
2,2018-08-08 08:38:49,2018-08-08,2018,8,8,2,3
3,2017-11-18 19:28:06,2017-11-18,2017,11,19,5,4
4,2018-02-13 21:18:39,2018-02-13,2018,2,21,1,1


In [6]:
# Paso 5: crear variables relacionadas con tiempos de aprobación y entrega

def crear_variables_logisticas(df):
    """
    Crea variables logísticas a partir de las fechas de pedidos.

    Parametros:
        df: DataFrame de pedidos.

    Devuelve:
        DataFrame con variables logísticas.
    """
    df = df.copy()

    df["approval_time_hours"] = (
        df["order_approved_at"] - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 3600

    df["carrier_delivery_days"] = (
        df["order_delivered_carrier_date"] - df["order_approved_at"]
    ).dt.total_seconds() / 86400

    df["customer_delivery_days"] = (
        df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 86400

    df["estimated_delivery_days"] = (
        df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 86400

    df["delay_days"] = (
        df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]
    ).dt.total_seconds() / 86400

    return df


orders = crear_variables_logisticas(orders)

orders[
    [
        "approval_time_hours",
        "carrier_delivery_days",
        "customer_delivery_days",
        "estimated_delivery_days",
        "delay_days"
    ]
].describe()

,approval_time_hours,carrier_delivery_days,customer_delivery_days,estimated_delivery_days,delay_days
count,99281.000000,97644.000000,96476.000000,99441.000000,96476.000000
mean,10.419094,2.805038,12.558702,23.767650,-11.179120
std,26.038004,3.549427,9.546530,8.832371,10.186113
min,0.000000,-171.219005,0.533414,1.648993,-146.016123
25%,0.215000,0.875509,6.766403,18.331690,-16.244384
50%,0.343333,1.818397,10.217755,23.240370,-11.948941
75%,14.580833,3.580469,15.720327,28.424861,-6.390000
max,4509.180556,125.762569,209.628611,155.135463,188.975081


In [7]:
# Paso 6: crear variables binarias relacionadas con el estado del pedido

def crear_variables_estado_pedido(df):
    """
    Crea variables binarias a partir del estado del pedido.

    Parametros:
        df: DataFrame de pedidos.

    Devuelve:
        DataFrame con variables binarias.
    """
    df = df.copy()

    df["is_delivered"] = np.where(
        df["order_status"] == "delivered",
        1,
        0
    )

    df["is_canceled"] = np.where(
        df["order_status"] == "canceled",
        1,
        0
    )

    df["is_unavailable"] = np.where(
        df["order_status"] == "unavailable",
        1,
        0
    )

    df["is_late"] = np.where(
        df["delay_days"] > 0,
        1,
        0
    )

    return df


orders = crear_variables_estado_pedido(orders)

orders[
    [
        "order_status",
        "is_delivered",
        "is_canceled",
        "is_unavailable",
        "is_late"
    ]
].head()

,order_status,is_delivered,is_canceled,is_unavailable,is_late
0,delivered,1,0,0,0
1,delivered,1,0,0,0
2,delivered,1,0,0,0
3,delivered,1,0,0,0
4,delivered,1,0,0,0


In [8]:
# Paso 7: preparar el dataset de festivos para unirlo con pedidos

holidays_merge = holidays.copy()

holidays_merge = holidays_merge.rename(
    columns={
        "holiday_date": "order_purchase_date",
        "holidayName": "holiday_name",
        "normalizeHolidayName": "holiday_name_normalized"
    }
)

holidays_merge["is_holiday"] = 1

holidays_merge[
    [
        "order_purchase_date",
        "holiday_name",
        "holiday_name_normalized",
        "is_holiday"
    ]
].head()

,order_purchase_date,holiday_name,holiday_name_normalized,is_holiday
596,2016-01-01,ano novo,ano novo,1
597,2016-02-09,carnaval,carnaval,1
598,2016-02-10,quarta-feira de cinzas (início da quaresma),quarta-feira de cinzas (início da quaresma),1
599,2016-03-25,sexta-feira santa,sexta-feira santa,1
600,2016-03-27,páscoa,páscoa,1


In [9]:
# Paso 8: unir pedidos con clientes mediante customer_id

df_orders_customers = orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

df_orders_customers.shape

(99441, 28)

In [10]:
# Paso 9: agregar información de items a nivel de pedido

def agregar_items_por_pedido(items):
    """
    Agrega las líneas de pedido a nivel order_id.

    Parametros:
        items: DataFrame de líneas de pedido.

    Devuelve:
        DataFrame agregado por pedido.
    """
    items_agg = (
        items
        .groupby("order_id", as_index=False)
        .agg(
            total_items=("order_item_id", "count"),
            total_products=("product_id", "nunique"),
            order_products_value=("price", "sum"),
            order_freight_value=("freight_value", "sum"),
            avg_item_price=("price", "mean"),
            max_item_price=("price", "max")
        )
    )

    return items_agg


items_agg = agregar_items_por_pedido(items)

items_agg.head()

,order_id,total_items,total_products,order_products_value,order_freight_value,avg_item_price,max_item_price
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,58.90,13.29,58.90,58.90
1,00018f77f2f0320c557190d7a144bdd3,1,1,239.90,19.93,239.90,239.90
2,000229ec398224ef6ca0657da4fc703e,1,1,199.00,17.87,199.00,199.00
3,00024acbcdf0a6daa1e931b038114c75,1,1,12.99,12.79,12.99,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,199.90,18.14,199.90,199.90


In [11]:
# Paso 10: agregar información de pagos a nivel de pedido

def agregar_pagos_por_pedido(payments):
    """
    Agrega los pagos a nivel order_id.

    Parametros:
        payments: DataFrame de pagos.

    Devuelve:
        DataFrame agregado por pedido.
    """
    payments_agg = (
        payments
        .groupby("order_id", as_index=False)
        .agg(
            total_payment_value=("payment_value", "sum"),
            payment_count=("payment_sequential", "count"),
            max_installments=("payment_installments", "max"),
            payment_type_main=("payment_type", lambda x: x.mode().iloc[0])
        )
    )

    return payments_agg


payments_agg = agregar_pagos_por_pedido(payments)

payments_agg.head()

,order_id,total_payment_value,payment_count,max_installments,payment_type_main
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3,credit_card


In [12]:
# Paso 11: añadir categoría de producto y traducción a las líneas de pedido

items_products = items.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one"
)

items_products = items_products.merge(
    categories,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

items_products.shape

(112650, 16)

In [13]:
# Paso 12: calcular la categoría principal de cada pedido según el mayor importe

def categoria_principal_por_pedido(items_products):
    """
    Obtiene la categoría principal de cada pedido según el mayor importe de producto.

    Parametros:
        items_products: DataFrame de items enriquecido con productos y categorías.

    Devuelve:
        DataFrame con order_id y categoría principal.
    """
    categoria_agg = (
        items_products
        .groupby(
            [
                "order_id",
                "product_category_name_english"
            ],
            as_index=False
        )
        .agg(
            category_value=("price", "sum"),
            category_items=("order_item_id", "count")
        )
    )

    categoria_agg = categoria_agg.sort_values(
        by=["order_id", "category_value"],
        ascending=[True, False]
    )

    categoria_principal = (
        categoria_agg
        .drop_duplicates(subset=["order_id"], keep="first")
        .rename(columns={
            "product_category_name_english": "main_product_category"
        })
    )

    return categoria_principal[
        [
            "order_id",
            "main_product_category",
            "category_value",
            "category_items"
        ]
    ]


category_main = categoria_principal_por_pedido(items_products)

category_main.head()

,order_id,main_product_category,category_value,category_items
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff,58.90,1
1,00018f77f2f0320c557190d7a144bdd3,pet_shop,239.90,1
2,000229ec398224ef6ca0657da4fc703e,furniture_decor,199.00,1
3,00024acbcdf0a6daa1e931b038114c75,perfumery,12.99,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools,199.90,1


In [14]:
# Paso 13: unir pedidos-clientes con items agregados, pagos y categoría principal

df_final = df_orders_customers.merge(
    items_agg,
    on="order_id",
    how="left",
    validate="one_to_one"
)

df_final = df_final.merge(
    payments_agg,
    on="order_id",
    how="left",
    validate="one_to_one"
)

df_final = df_final.merge(
    category_main,
    on="order_id",
    how="left",
    validate="one_to_one"
)

df_final.shape

(99441, 41)

In [15]:
# Paso 14: unir dataset final con festivos usando la fecha de compra

df_final = df_final.merge(
    holidays_merge[
        [
            "order_purchase_date",
            "holiday_name",
            "holiday_name_normalized",
            "is_holiday"
        ]
    ],
    on="order_purchase_date",
    how="left",
    validate="many_to_one"
)

df_final["is_holiday"] = df_final["is_holiday"].fillna(0).astype(int)

df_final[
    [
        "order_purchase_date",
        "holiday_name",
        "is_holiday"
    ]
].head()

,order_purchase_date,holiday_name,is_holiday
0,2017-10-02,NaN,0
1,2018-07-24,NaN,0
2,2018-08-08,NaN,0
3,2017-11-18,NaN,0
4,2018-02-13,carnaval,1


In [16]:
# Comprobar cuántos pedidos coinciden con festivos

df_final["is_holiday"].value_counts()

is_holiday
0    96739
1     2702
Name: count, dtype: int64

In [17]:
# Ver pedidos realizados en festivos

df_final.loc[
    df_final["is_holiday"] == 1,
    ["order_purchase_date", "holiday_name", "is_holiday"]
].head(20)

,order_purchase_date,holiday_name,is_holiday
4,2018-02-13,carnaval,1
62,2017-11-15,proclamação da república,1
90,2018-05-01,dia mundial do trabalho,1
152,2018-02-14,quarta-feira de cinzas (início da quaresma),1
166,2017-11-15,proclamação da república,1
250,2018-03-30,sexta-feira santa,1
342,2018-04-01,páscoa,1
391,2017-04-14,sexta-feira santa,1
446,2017-10-12,nossa senhora aparecida,1
488,2017-10-12,nossa senhora aparecida,1


In [18]:
# Revisar fechas de festivos del dataset holidays

holidays_merge[
    ["order_purchase_date", "holiday_name"]
].sort_values("order_purchase_date")

,order_purchase_date,holiday_name
596,2016-01-01,ano novo
597,2016-02-09,carnaval
598,2016-02-10,quarta-feira de cinzas (início da quaresma)
599,2016-03-25,sexta-feira santa
600,2016-03-27,páscoa
601,2016-04-21,tiradentes
602,2016-05-01,dia mundial do trabalho
603,2016-05-26,corpus christi
604,2016-09-07,independência do brasil
605,2016-10-12,nossa senhora aparecida


In [19]:
print(df_final["order_purchase_date"].dtype)
print(holidays_merge["order_purchase_date"].dtype)

datetime64[ns]
datetime64[ns]


In [20]:
# Paso 15: crear variables económicas derivadas

def crear_variables_economicas(df):
    """
    Crea variables economicas finales para el analisis.

    Parametros:
        df: DataFrame final.

    Devuelve:
        DataFrame con variables economicas derivadas.
    """
    df = df.copy()

    df["order_total_value"] = (
        df["order_products_value"]
        + df["order_freight_value"]
    )

    df["freight_ratio"] = np.where(
        df["order_total_value"] > 0,
        df["order_freight_value"] / df["order_total_value"],
        np.nan
    )

    df["avg_product_value_per_item"] = np.where(
        df["total_items"] > 0,
        df["order_products_value"] / df["total_items"],
        np.nan
    )

    return df


df_final = crear_variables_economicas(df_final)

df_final[
    [
        "order_products_value",
        "order_freight_value",
        "order_total_value",
        "freight_ratio",
        "avg_product_value_per_item"
    ]
].head()

,order_products_value,order_freight_value,order_total_value,freight_ratio,avg_product_value_per_item
0,29.99,8.72,38.71,0.225265,29.99
1,118.70,22.76,141.46,0.160894,118.70
2,159.90,19.22,179.12,0.107302,159.90
3,45.00,27.20,72.20,0.376731,45.00
4,19.90,8.72,28.62,0.304682,19.90


In [21]:
# Paso 16: crear variables de segmentación para enriquecer el análisis

def crear_segmentacion_pedido(df):
    """
    Crea segmentos de pedido según valor y número de productos.

    Parametros:
        df: DataFrame final.

    Devuelve:
        DataFrame con variables de segmentación.
    """
    df = df.copy()

    df["order_value_segment"] = pd.cut(
        df["order_total_value"],
        bins=[-0.01, 50, 150, 500, np.inf],
        labels=[
            "bajo",
            "medio",
            "alto",
            "muy_alto"
        ]
    )

    df["items_segment"] = pd.cut(
        df["total_items"],
        bins=[-0.01, 1, 3, np.inf],
        labels=[
            "un_producto",
            "dos_tres_productos",
            "mas_de_tres_productos"
        ]
    )

    return df


df_final = crear_segmentacion_pedido(df_final)

df_final[
    [
        "order_total_value",
        "order_value_segment",
        "total_items",
        "items_segment"
    ]
].head()

,order_total_value,order_value_segment,total_items,items_segment
0,38.71,bajo,1.0,un_producto
1,141.46,medio,1.0,un_producto
2,179.12,alto,1.0,un_producto
3,72.20,medio,1.0,un_producto
4,28.62,bajo,1.0,un_producto


In [22]:
# Paso 17: convertir contadores a enteros nullable

def convertir_contadores_enteros(df):
    """
    Convierte variables de conteo a tipo entero nullable.
    """

    df = df.copy()

    columnas = [
        "total_items",
        "total_products",
        "payment_count",
        "max_installments"
    ]

    for columna in columnas:
        df[columna] = df[columna].astype("Int64")

    return df


df_final = convertir_contadores_enteros(df_final)

In [23]:
# Paso 18: comprobar que el dataset final mantiene una fila por pedido

duplicados_order_id = df_final["order_id"].duplicated().sum()

print("Duplicados por order_id:", duplicados_order_id)

Duplicados por order_id: 0


In [24]:
df_final.loc[
    df_final["total_items"].isna(),
    "order_status"
].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [25]:
# Paso 19: comprobar que se cumplen los requisitos mínimos de filas y columnas

cumple_filas = df_final.shape[0] >= 50000
cumple_columnas = df_final.shape[1] >= 20

print("Cumple mínimo de 50.000 filas:", cumple_filas)
print("Cumple mínimo de 20 columnas:", cumple_columnas)

Cumple mínimo de 50.000 filas: True
Cumple mínimo de 20 columnas: True


In [26]:
# Paso 20: revisar tipos de datos del dataset final

tipos_finales = pd.DataFrame({
    "columna": df_final.columns,
    "tipo_dato": df_final.dtypes.astype(str).values
})

tipos_finales

,columna,tipo_dato
0,order_id,object
1,customer_id,object
2,order_status,object
3,order_purchase_timestamp,datetime64[ns]
4,order_approved_at,datetime64[ns]
5,order_delivered_carrier_date,datetime64[ns]
6,order_delivered_customer_date,datetime64[ns]
7,order_estimated_delivery_date,datetime64[ns]
8,order_purchase_date,datetime64[ns]
9,purchase_year,int32


In [27]:
# Paso 21: revisar fechas y nombres de festivos tras la conversion

df_final.loc[
    df_final["is_holiday"] == 1,
    ["order_purchase_date", "holiday_name"]
].drop_duplicates().sort_values("order_purchase_date")

,order_purchase_date,holiday_name
1390,2017-02-28,carnaval
807,2017-03-01,quarta-feira de cinzas (início da quaresma)
391,2017-04-14,sexta-feira santa
7527,2017-04-16,páscoa
2447,2017-04-21,tiradentes
1239,2017-05-01,dia mundial do trabalho
921,2017-06-15,corpus christi
2344,2017-09-07,independência do brasil
446,2017-10-12,nossa senhora aparecida
498,2017-11-02,finados


In [28]:
# Revisar fechas y nombres de festivos antes del rename

holidays[
    [
        "holiday_date",
        "holidayName"
    ]
].sort_values("holiday_date")

,holiday_date,holidayName
596,2016-01-01,ano novo
597,2016-02-09,carnaval
598,2016-02-10,quarta-feira de cinzas (início da quaresma)
599,2016-03-25,sexta-feira santa
600,2016-03-27,páscoa
601,2016-04-21,tiradentes
602,2016-05-01,dia mundial do trabalho
603,2016-05-26,corpus christi
604,2016-09-07,independência do brasil
605,2016-10-12,nossa senhora aparecida


Durante la validación de la tabla de festivos se detectó una interpretación incorrecta del formato de fecha. Las fechas originales estaban almacenadas en formato día/mes/año y Pandas las había interpretado inicialmente como mes/día/año, provocando una asignación errónea de algunos festivos. Se corrigió la conversión utilizando dayfirst=True y se validaron manualmente los principales festivos nacionales de Brasil para los años 2016, 2017 y 2018, verificando que las fechas fueran coherentes antes de realizar la unión con los pedidos.

In [29]:
# Paso 22: validar rangos de fechas principales

columnas_fecha = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "order_purchase_date"
]

for columna in columnas_fecha:
    print(columna)
    print("min:", df_final[columna].min())
    print("max:", df_final[columna].max())
    print()

order_purchase_timestamp
min: 2016-09-04 21:15:19
max: 2018-10-17 17:30:18

order_approved_at
min: 2016-09-15 12:16:38
max: 2018-09-03 17:40:06

order_delivered_carrier_date
min: 2016-10-08 10:34:01
max: 2018-09-11 19:48:28

order_delivered_customer_date
min: 2016-10-11 13:46:32
max: 2018-10-17 13:22:46

order_estimated_delivery_date
min: 2016-09-30 00:00:00
max: 2018-11-12 00:00:00

order_purchase_date
min: 2016-09-04 00:00:00
max: 2018-10-17 00:00:00



Los rangos temporales de todas las variables de fecha son coherentes con el período cubierto por el dataset (septiembre de 2016 a octubre de 2018). No se detectaron fechas fuera del rango esperado ni valores anómalos que sugieran errores de importación o conversión. La fecha máxima estimada de entrega se extiende hasta noviembre de 2018, lo cual resulta lógico al corresponder a pedidos realizados en las últimas semanas del período analizado.

In [30]:
# Paso 23: revisar tiempos logisticos negativos

columnas_logisticas = [
    "approval_time_hours",
    "carrier_delivery_days",
    "customer_delivery_days",
    "estimated_delivery_days",
    "delay_days"
]

for columna in columnas_logisticas:
    negativos = (df_final[columna] < 0).sum()
    print(f"{columna}: {negativos} valores negativos")

approval_time_hours: 0 valores negativos
carrier_delivery_days: 1359 valores negativos
customer_delivery_days: 0 valores negativos
estimated_delivery_days: 0 valores negativos
delay_days: 88649 valores negativos


In [31]:
df_final.loc[
    df_final["carrier_delivery_days"] < 0,
    [
        "order_id",
        "order_approved_at",
        "order_delivered_carrier_date",
        "carrier_delivery_days"
    ]
].head(20)

,order_id,order_approved_at,order_delivered_carrier_date,carrier_delivery_days
15,dcb36b511fcac050b97cd5c05de84dc3,2018-06-12 23:31:02,2018-06-11 14:54:00,-1.359051
64,688052146432ef8253587b930b01a06d,2018-04-24 18:25:22,2018-04-23 19:19:14,-0.962593
199,58d4c4747ee059eeeb865b349b41f53a,2018-07-26 23:31:53,2018-07-24 12:57:00,-2.440891
210,412fccb2b44a99b36714bca3fef8ad7b,2018-07-23 12:31:53,2018-07-23 12:24:00,-0.005475
415,56a4ac10a4a8f2ba7693523bb439eede,2018-07-27 23:31:09,2018-07-24 14:03:00,-3.394549
481,32e4fa9bb468884309b58b9348de70c3,2018-07-05 16:33:06,2018-07-05 14:50:00,-0.071597
483,4df92d82d79c3b52c7138679fa9b07fc,2018-07-29 23:30:52,2018-07-26 14:46:00,-3.364491
585,16e38caa92e342c7780f68832f832d4d,2018-05-07 16:52:39,2018-05-07 15:09:00,-0.071979
615,b9afddbdcfadc9a87b41a83271c3e888,2018-08-16 14:05:13,2018-08-16 13:27:00,-0.026539
817,6051e6d3da9a50b7325cbe9c81025062,2018-07-05 16:31:26,2018-07-04 12:14:00,-1.178773


In [32]:
# Ver registros con tiempo logistico negativo

df_final.loc[
    df_final["carrier_delivery_days"] < 0,
    [
        "order_id",
        "order_status",
        "order_approved_at",
        "order_delivered_carrier_date",
        "carrier_delivery_days"
    ]
].head(20)

,order_id,order_status,order_approved_at,order_delivered_carrier_date,carrier_delivery_days
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-12 23:31:02,2018-06-11 14:54:00,-1.359051
64,688052146432ef8253587b930b01a06d,delivered,2018-04-24 18:25:22,2018-04-23 19:19:14,-0.962593
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-26 23:31:53,2018-07-24 12:57:00,-2.440891
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-23 12:31:53,2018-07-23 12:24:00,-0.005475
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-27 23:31:09,2018-07-24 14:03:00,-3.394549
481,32e4fa9bb468884309b58b9348de70c3,delivered,2018-07-05 16:33:06,2018-07-05 14:50:00,-0.071597
483,4df92d82d79c3b52c7138679fa9b07fc,delivered,2018-07-29 23:30:52,2018-07-26 14:46:00,-3.364491
585,16e38caa92e342c7780f68832f832d4d,delivered,2018-05-07 16:52:39,2018-05-07 15:09:00,-0.071979
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 14:05:13,2018-08-16 13:27:00,-0.026539
817,6051e6d3da9a50b7325cbe9c81025062,delivered,2018-07-05 16:31:26,2018-07-04 12:14:00,-1.178773


In [33]:
df_final["carrier_before_approval"] = (
    df_final["carrier_delivery_days"] < 0
).astype(int)

In [34]:
# Porcentaje de registros afectados

(
    df_final["carrier_before_approval"]
    .value_counts(normalize=True)
    * 100
).round(2)

carrier_before_approval
0    98.63
1     1.37
Name: proportion, dtype: float64

Se identificaron 1.359 pedidos (1,37 % del total) cuya fecha de entrega al transportista es anterior a la fecha de aprobación del pedido. Tras revisar una muestra de registros, se observó que algunas diferencias corresponden a minutos u horas, mientras que otras alcanzan varios días. Dado que representan un porcentaje reducido del conjunto de datos y corresponden a registros originales de la fuente, se decidió conservarlos y crear una variable indicadora (carrier_before_approval) para facilitar su identificación en análisis posteriores.

In [35]:
# Paso 24: revisar valores negativos o incoherentes en variables economicas

columnas_economicas = [
    "order_products_value",
    "order_freight_value",
    "total_payment_value",
    "order_total_value",
    "freight_ratio",
    "avg_product_value_per_item"
]

for columna in columnas_economicas:
    negativos = (df_final[columna] < 0).sum()
    print(f"{columna}: {negativos} valores negativos")

order_products_value: 0 valores negativos
order_freight_value: 0 valores negativos
total_payment_value: 0 valores negativos
order_total_value: 0 valores negativos
freight_ratio: 0 valores negativos
avg_product_value_per_item: 0 valores negativos


Se validaron las principales variables económicas derivadas del proceso de agregación y transformación. No se detectaron importes negativos ni inconsistencias de signo en los valores de productos, costes de envío, pagos totales, importe total del pedido o métricas derivadas. Esto confirma la correcta construcción de las variables económicas utilizadas posteriormente en el análisis exploratorio y estadístico.

In [36]:
# Paso 25: comparar total calculado del pedido con total pagado

df_final["payment_difference"] = (
    df_final["total_payment_value"] - df_final["order_total_value"]
)

df_final["payment_difference"].describe()

count    98665.000000
mean         0.029092
std          1.129221
min        -51.620000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: payment_difference, dtype: float64

In [37]:
(df_final["payment_difference"] != 0).sum()

np.int64(20390)

In [38]:
(
    df_final["payment_difference"] != 0
).mean() * 100

np.float64(20.50462083044217)

20,50 % de los pedidos presentan alguna diferencia
entre el importe calculado y el importe pagado.

In [39]:
df_final[
    [
        "order_id",
        "order_total_value",
        "total_payment_value",
        "payment_difference",
        "payment_type_main"
    ]
].sort_values(
    "payment_difference",
    ascending=False
).head(20)

,order_id,order_total_value,total_payment_value,payment_difference,payment_type_main
11791,ce6d150fb29ada17d2082f4847107665,1403.66,1586.47,182.81,credit_card
48686,6e5fe7366a2e1bfbf3257dba0af1267f,287.91,406.92,119.01,credit_card
70865,70b742795bc441e94a44a084b6d9ce7a,466.93,578.82,111.89,credit_card
33150,996c7e73600ad3723e8627ab7bef81e4,587.90,664.43,76.53,credit_card
52985,70b7e94ea46d3e8b5bc12a50186edaf0,213.15,274.84,61.69,credit_card
85434,bc2c82b0ef78d2252b6176d1972db7c9,242.01,303.02,61.01,credit_card
94304,af9ffff2ce6b3defd34fd4c78857a379,413.17,466.97,53.80,credit_card
68811,bfdb5bbb06458d600a33d61f5f287472,348.93,394.36,45.43,credit_card
8548,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,254.45,293.89,39.44,credit_card
2615,b7579d24f5b2dd3e20f2e57d0e07d170,466.28,504.37,38.09,credit_card


In [40]:
df_final[
    df_final["order_total_value"] == 0
][
    [
        "order_id",
        "total_items",
        "order_products_value",
        "order_freight_value",
        "order_total_value",
        "total_payment_value"
    ]
].head(20)

,order_id,total_items,order_products_value,order_freight_value,order_total_value,total_payment_value


In [41]:
(
    df_final["order_total_value"] == 0
).sum()

np.int64(0)

In [42]:
df_final["order_total_value"].isna().sum()

np.int64(775)

In [43]:
df_final[
    [
        "order_id",
        "order_total_value",
        "total_payment_value",
        "payment_difference",
        "payment_type_main"
    ]
].sort_values(
    "payment_difference",
    ascending=True
).head(20)

,order_id,order_total_value,total_payment_value,payment_difference,payment_type_main
31661,262118ce178bb3e4590a3adcf6d62e6b,177.74,126.12,-51.62,credit_card
45606,fd33085945f15975375cd8ec85440511,234.62,212.82,-21.80,credit_card
1986,6e57e23ecac1ae881286657694444267,350.41,333.91,-16.50,debit_card
35623,4154bf1348caac78152fe76e3e9c4af8,165.26,150.27,-14.99,credit_card
61544,6dcf0aeb8b1eb4021c26e1d0e9394979,333.92,318.97,-14.95,debit_card
72643,0e556f5eafbf3eb399290101b183b10e,95.66,81.90,-13.76,credit_card
51874,8092da256aefda13b330290d2ca86521,115.65,105.26,-10.39,credit_card
10250,aa6bd33ba1853d846d3085a88ae37083,35.14,25.14,-10.00,credit_card
91801,4387477eec4b3c89b39f3f454940d059,231.92,222.02,-9.90,debit_card
70058,c6f6cbb3c845593222d5c594bb69c5fd,113.33,104.34,-8.99,credit_card


In [44]:
# Revisar pedidos con diferencias relevantes entre pedido y pago

df_final.loc[
    df_final["payment_difference"].abs() > 1,
    [
        "order_id",
        "order_status",
        "order_total_value",
        "total_payment_value",
        "payment_difference"
    ]
].head(20)

,order_id,order_status,order_total_value,total_payment_value,payment_difference
1080,84d6d9710c8af32b5e88f2d1c14ab871,delivered,57.68,61.70,4.02
1126,74016effecaa79d592487f6a4ee47d4b,delivered,51.51,56.96,5.45
1669,239f380355f65dcb68551f07d16fc4a8,delivered,222.63,251.63,29.00
1729,4c57f545143e8865ca2347d8cba154a7,delivered,139.61,151.01,11.40
1986,6e57e23ecac1ae881286657694444267,delivered,350.41,333.91,-16.50
2357,051fcda88d997d3ff86012da2a556342,delivered,56.60,51.70,-4.90
2541,8b5058499c412c6cf8d013de40e4f9d2,delivered,48.77,51.02,2.25
2615,b7579d24f5b2dd3e20f2e57d0e07d170,delivered,466.28,504.37,38.09
3077,8b7fd198ad184563c231653673e75a7f,delivered,49.33,56.97,7.64
4663,3b773756d8fc59a7cd1a0c4ac584e397,delivered,167.17,188.96,21.79


In [45]:
df_final["payment_difference"].abs().describe()

count    98665.000000
mean         0.033162
std          1.129109
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: payment_difference, dtype: float64

Se comparó el importe total calculado a partir de los productos y costes de envío con el importe total registrado en los pagos. Aproximadamente un 20,5 % de los pedidos presentan alguna diferencia entre ambos valores. Tras revisar una muestra de registros, se observó que las discrepancias suelen ser reducidas y compatibles con descuentos, cupones, pagos fraccionados o ajustes propios de la plataforma de comercio electrónico. No se identificaron errores sistemáticos en el proceso de agregación, por lo que los registros fueron conservados para el análisis posterior.

Se comparó el importe total calculado del pedido con el importe total registrado en los pagos. Tras corregir el tratamiento de los pedidos sin información de artículos, las diferencias observadas resultaron mínimas. La diferencia absoluta media fue de 0,03 reales brasileños y el 75 % de los pedidos presentaron una diferencia igual a cero. Estos resultados indican una elevada coherencia entre la información de pedidos y pagos, por lo que no fue necesario realizar ajustes adicionales sobre estas variables.

In [46]:
# Paso 26: validar que variables binarias solo contienen 0 y 1

columnas_binarias = [
    "is_delivered",
    "is_canceled",
    "is_unavailable",
    "is_late",
    "is_holiday"
]

for columna in columnas_binarias:
    print(columna)
    print(df_final[columna].value_counts(dropna=False))
    print()

is_delivered
is_delivered
1    96478
0     2963
Name: count, dtype: int64

is_canceled
is_canceled
0    98816
1      625
Name: count, dtype: int64

is_unavailable
is_unavailable
0    98832
1      609
Name: count, dtype: int64

is_late
is_late
0    91614
1     7827
Name: count, dtype: int64

is_holiday
is_holiday
0    96739
1     2702
Name: count, dtype: int64



In [47]:
# Paso 27: revisar categorias principales

df_final["main_product_category"].value_counts(dropna=False).head(20)

main_product_category
bed_bath_table              9350
health_beauty               8802
sports_leisure              7682
computers_accessories       6671
furniture_decor             6332
housewares                  5821
watches_gifts               5606
telephony                   4179
auto                        3879
toys                        3871
cool_stuff                  3602
garden_tools                3476
perfumery                   3146
baby                        2833
electronics                 2536
stationery                  2290
fashion_bags_accessories    1855
pet_shop                    1703
without_category            1410
office_furniture            1270
Name: count, dtype: int64

In [48]:
# Paso 28: validar duplicados por clave principal

print("Duplicados por order_id:", df_final["order_id"].duplicated().sum())
print("Filas totales:", df_final.shape[0])
print("Pedidos unicos:", df_final["order_id"].nunique())

Duplicados por order_id: 0
Filas totales: 99441
Pedidos unicos: 99441


In [49]:
# Paso 29: validar requisitos minimos del proyecto

print("Filas:", df_final.shape[0])
print("Columnas:", df_final.shape[1])
print("Cumple 50000 filas:", df_final.shape[0] >= 50000)
print("Cumple 20 columnas:", df_final.shape[1] >= 20)

Filas: 99441
Columnas: 51
Cumple 50000 filas: True
Cumple 20 columnas: True


In [50]:
df_final["customer_zip_code_prefix"].head()

0     3149
1    47813
2    75265
3    59296
4     9195
Name: customer_zip_code_prefix, dtype: object

In [51]:
# Paso 30: ajuste final de tipos de datos

def ajustar_tipos_finales(df):
    """
    Ajusta los tipos de datos finales del dataset
    antes de su exportación para análisis y Power BI.

    Parámetros:
        df (pd.DataFrame): Dataset final.

    Devuelve:
        pd.DataFrame: Dataset con tipos corregidos.
    """

    df = df.copy()

    columnas_enteras = [
        "total_items",
        "total_products",
        "payment_count",
        "max_installments",
        "category_items"
    ]

    for columna in columnas_enteras:
        df[columna] = df[columna].astype("Int64")

    df["payment_difference"] = (
        df["payment_difference"]
        .round(2)
    )

    return df


df_final = ajustar_tipos_finales(df_final)

In [52]:
df_final[
    [
        "total_items",
        "total_products",
        "payment_count",
        "max_installments",
        "category_items",
        "payment_difference"
    ]
].dtypes

total_items             Int64
total_products          Int64
payment_count           Int64
max_installments        Int64
category_items          Int64
payment_difference    float64
dtype: object

In [53]:
df_final[
    [
        "total_items",
        "total_products",
        "payment_count",
        "max_installments",
        "category_items",
        "payment_difference"
    ]
].head()

,total_items,total_products,payment_count,max_installments,category_items,payment_difference
0,1,1,3,1,1,0.0
1,1,1,1,1,1,0.0
2,1,1,1,3,1,0.0
3,1,1,1,1,1,0.0
4,1,1,1,1,1,0.0


In [54]:
# Paso 31: revisar nulos del dataset final

def resumen_nulos_final(df):
    """
    Calcula valores nulos del dataset final.

    Parametros:
        df: DataFrame final.

    Devuelve:
        DataFrame con resumen de nulos.
    """
    resumen = pd.DataFrame({
        "columna": df.columns,
        "nulos": df.isna().sum().values,
        "porcentaje_nulos": (df.isna().mean().values * 100).round(2)
    })

    return resumen.sort_values(by="porcentaje_nulos", ascending=False)


nulos_final = resumen_nulos_final(df_final)

nulos_final[nulos_final["nulos"] > 0].head(20)

,columna,nulos,porcentaje_nulos
42,holiday_name_normalized,96739,97.28
41,holiday_name,96739,97.28
6,order_delivered_customer_date,2965,2.98
17,customer_delivery_days,2965,2.98
19,delay_days,2965,2.98
16,carrier_delivery_days,1797,1.81
5,order_delivered_carrier_date,1783,1.79
38,main_product_category,796,0.80
40,category_items,796,0.80
39,category_value,796,0.80


In [55]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 51 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
 8   order_purchase_date            99441 non-null  datetime64[ns]
 9   purchase_year                  99441 non-null  int32         
 10  purchase_month                 99441 non-null  int32         
 11  purchase_day   

In [56]:
df_final.head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_purchase_date,purchase_year,purchase_month,purchase_day,purchase_hour,purchase_dayofweek,purchase_quarter,approval_time_hours,carrier_delivery_days,customer_delivery_days,estimated_delivery_days,delay_days,is_delivered,is_canceled,is_unavailable,is_late,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_items,total_products,order_products_value,order_freight_value,avg_item_price,max_item_price,total_payment_value,payment_count,max_installments,payment_type_main,main_product_category,category_value,category_items,holiday_name,holiday_name_normalized,is_holiday,order_total_value,freight_ratio,avg_product_value_per_item,order_value_segment,items_segment,carrier_before_approval,payment_difference
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017-10-02,2017,10,2,10,0,4,0.178333,2.366493,8.436574,15.544063,-7.107488,1,0,0,0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,sp,1,1,29.99,8.72,29.99,29.99,38.71,3,1,voucher,housewares,29.99,1,NaN,NaN,0,38.71,0.225265,29.99,bajo,un_producto,0,0.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018-07-24,2018,7,24,20,1,3,30.713889,0.462882,13.782037,19.137766,-5.355729,1,0,0,0,af07308b275d755c9edb36a90c618231,47813,barreiras,ba,1,1,118.70,22.76,118.70,118.70,141.46,1,1,boleto,perfumery,118.70,1,NaN,NaN,0,141.46,0.160894,118.70,medio,un_producto,0,0.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018-08-08,2018,8,8,8,2,3,0.276111,0.204595,9.394213,26.639711,-17.245498,1,0,0,0,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,go,1,1,159.90,19.22,159.90,159.90,179.12,1,3,credit_card,auto,159.90,1,NaN,NaN,0,179.12,0.107302,159.90,alto,un_producto,0,0.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017-11-18,2017,11,18,19,5,4,0.298056,3.745833,13.208750,26.188819,-12.980069,1,0,0,0,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,rn,1,1,45.00,27.20,45.00,45.00,72.20,1,1,credit_card,pet_shop,45.00,1,NaN,NaN,0,72.20,0.376731,45.00,medio,un_producto,0,0.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018-02-13,2018,2,13,21,1,1,1.030556,0.893113,2.873877,12.112049,-9.238171,1,0,0,0,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,sp,1,1,19.90,8.72,19.90,19.90,28.62,1,1,credit_card,stationery,19.90,1,carnaval,carnaval,1,28.62,0.304682,19.90,bajo,un_producto,0,0.0
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01,2017-07-09,2017,7,9,21,6,3,0.218889,1.699896,16.542245,22.085359,-5.543113,1,0,0,0,80bb27c7c16e8f973207a5086ab329e2,86320,congonhinhas,pr,1,1,147.90,27.36,147.90,147.90,175.26,1,6,credit_card,auto,147.90,1,NaN,NaN,0,175.26,0.156111,147.90,alto,un_producto,0,0.0
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaT,NaT,2017-05-09,2017-04-11,2017,4,11,12,1,2,49.052500,NaN,NaN,27.484630,NaN,0,0,0,0,36edbb3fb164b1f16485364b6fb04c73,98900,santa rosa,rs,1,1,49.90,16.05,49.90,49.90,65.95,1,1,credit_card,without_category,49.90,1,NaN,NaN,0,65.95,0.243366,49.90,medio,un_producto,0,0.0
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2

In [57]:
# Paso 32: guardar el dataset final limpio y enriquecido

df_final.to_csv(
    RUTA_PROCESSED / "ecommerce_brazil_final.csv",
    index=False
)

In [58]:
# Paso 33: guardar el dataset final limpio y enriquecido en formato pickle

df_final.to_pickle(
    RUTA_PROCESSED / "ecommerce_brazil_final.pkl"
)

# Conclusiones del Notebook 03 - Feature Engineering

## Objetivo del notebook

El objetivo de este notebook ha sido construir un dataset analítico único a partir de las distintas tablas del proyecto Olist, integrando información de pedidos, clientes, productos, pagos y festivos. Además, se han generado nuevas variables que permiten enriquecer el análisis posterior desde una perspectiva temporal, logística, económica y de comportamiento de compra.

El resultado final es un dataset consolidado de **99.441 pedidos** y **51 variables**, preparado para realizar el análisis exploratorio y estadístico de los siguientes notebooks.

---

# Integración de datasets

Durante este proceso se realizó la unión de las distintas fuentes de información utilizando las claves de negocio identificadas en la fase de exploración y limpieza.

Las principales relaciones utilizadas fueron:

| Tabla origen | Tabla destino      | Clave               |
| ------------ | ------------------ | ------------------- |
| Orders       | Customers          | customer_id         |
| Orders       | Items agregados    | order_id            |
| Orders       | Payments agregados | order_id            |
| Orders       | Products agregados | order_id            |
| Orders       | Holidays           | order_purchase_date |

La integración permitió construir una visión completa de cada pedido incorporando información del cliente, características de los productos, métodos de pago y contexto temporal asociado a festivos nacionales de Brasil.

---

# Variables creadas durante el proceso de Feature Engineering

Se generaron nuevas variables para enriquecer el análisis.

## Variables temporales

A partir de la fecha de compra se crearon:

* purchase_year
* purchase_month
* purchase_day
* purchase_hour
* purchase_dayofweek
* purchase_quarter

Estas variables permitirán analizar patrones de compra según estacionalidad, día de la semana o franja horaria.

---

## Variables logísticas

Se calcularon métricas relacionadas con el ciclo de vida del pedido:

* approval_time_hours
* carrier_delivery_days
* customer_delivery_days
* estimated_delivery_days
* delay_days

Estas variables permiten evaluar la eficiencia logística y los tiempos de entrega.

---

## Variables económicas

Se generaron métricas económicas agregadas:

* order_total_value
* freight_ratio
* avg_product_value_per_item
* payment_difference

Estas variables facilitan el análisis del valor económico de los pedidos y la relación entre productos, costes de envío y pagos realizados.

---

## Variables binarias

Se crearon indicadores para simplificar análisis posteriores:

* is_delivered
* is_canceled
* is_unavailable
* is_late
* is_holiday
* carrier_before_approval

Estas variables permiten segmentar fácilmente los pedidos según diferentes condiciones operativas.

---

## Variables de segmentación

Para facilitar el análisis exploratorio se crearon segmentos:

* order_value_segment
* items_segment

Estas variables permitirán estudiar diferencias de comportamiento según el valor económico del pedido y el volumen de artículos comprados.

---

# Validaciones realizadas

Tras la creación de variables se ejecutó un proceso exhaustivo de validación para garantizar la calidad del dataset final.

---

## Validación de fechas

Se verificó que todas las variables temporales presentaran rangos coherentes con el periodo cubierto por el dataset:

* Fecha mínima de compra: 04/09/2016
* Fecha máxima de compra: 17/10/2018

No se detectaron fechas fuera de rango ni errores de conversión.

También se revisó la coherencia de los festivos nacionales de Brasil, corrigiendo inicialmente un problema de interpretación del formato de fecha que provocaba asignaciones incorrectas de algunos festivos.

---

## Validación de variables logísticas

Se analizaron los tiempos calculados para detectar valores incoherentes.

Los resultados mostraron:

* 0 pedidos aprobados antes de ser comprados.
* 0 pedidos entregados antes de ser enviados.
* 0 fechas estimadas anteriores a la compra.

Se detectaron 1.359 pedidos (1,37 %) cuya fecha de entrega al transportista era anterior a la fecha de aprobación. Tras revisar una muestra de registros se observó que la mayoría corresponden a pequeñas diferencias horarias o posibles inconsistencias operativas del sistema original, por lo que se decidió conservarlos y documentarlos mediante la variable `carrier_before_approval`.

---

## Validación económica

Se comprobó la coherencia entre:

* valor de productos
* costes de envío
* pagos registrados
* importe total del pedido

No se detectaron importes negativos en ninguna de las variables económicas.

Inicialmente se identificaron discrepancias elevadas entre el importe calculado del pedido y el importe pagado. Tras la investigación se comprobó que estas diferencias procedían de pedidos sin información en la tabla de artículos (`items`).

Para evitar interpretaciones incorrectas se modificó el cálculo de `order_total_value`, manteniendo los valores nulos cuando no existía información suficiente para calcular el importe del pedido.

Tras esta corrección:

* El 75 % de los pedidos presentan diferencia igual a cero.
* La diferencia absoluta media es de únicamente 0,03 reales brasileños.

Esto confirma una elevada coherencia entre los pedidos y los pagos registrados.

---

## Validación de variables binarias

Se verificó que todas las variables binarias únicamente contuvieran los valores:

* 0
* 1

Además, las distribuciones obtenidas fueron coherentes con los resultados observados en la fase de exploración inicial.

---

# Tratamiento de los valores nulos

Los valores nulos existentes en el dataset final fueron analizados individualmente.

Se decidió conservarlos cuando representaban situaciones reales del negocio y su imputación podía introducir sesgos o interpretaciones erróneas.

### Casos conservados

#### Festivos

Los nulos de:

* holiday_name
* holiday_name_normalized

indican simplemente que el pedido no se realizó en un día festivo.

Esta información ya queda representada mediante la variable binaria `is_holiday`.

---

#### Fechas de entrega

Los nulos en:

* order_delivered_customer_date
* customer_delivery_days
* delay_days

corresponden principalmente a pedidos cancelados, no disponibles o no entregados.

Por tanto, representan información real y no errores de calidad de datos.

---

#### Variables económicas y de productos

Los nulos en:

* total_items
* total_products
* order_products_value
* order_freight_value
* order_total_value
* main_product_category

corresponden a pedidos sin información disponible en la tabla de artículos (`items`).

En estos casos se decidió mantener los valores nulos ya que sustituirlos por cero implicaría asumir incorrectamente que el pedido tiene valor cero cuando en realidad el valor es desconocido.

---

# Estructura del dataset final

El dataset final contiene tres tipos de variables claramente diferenciadas.

## Variables originales

Proceden directamente de las tablas fuente:

### Orders

* order_id
* customer_id
* order_status
* order_purchase_timestamp
* order_approved_at
* order_delivered_carrier_date
* order_delivered_customer_date
* order_estimated_delivery_date

### Customers

* customer_unique_id
* customer_zip_code_prefix
* customer_city
* customer_state

### Holidays

* holiday_name
* holiday_name_normalized

---

## Variables agregadas

Proceden de procesos de agrupación y resumen sobre tablas relacionadas.

### Items

* total_items
* total_products
* order_products_value
* order_freight_value
* avg_item_price
* max_item_price

### Payments

* total_payment_value
* payment_count
* max_installments
* payment_type_main

### Products

* main_product_category
* category_value
* category_items

---

## Variables derivadas

Han sido creadas durante el proceso de Feature Engineering.

### Temporales

* purchase_year
* purchase_month
* purchase_day
* purchase_hour
* purchase_dayofweek
* purchase_quarter

### Logísticas

* approval_time_hours
* carrier_delivery_days
* customer_delivery_days
* estimated_delivery_days
* delay_days

### Económicas

* order_total_value
* freight_ratio
* avg_product_value_per_item
* payment_difference

### Indicadoras

* is_delivered
* is_canceled
* is_unavailable
* is_late
* is_holiday
* carrier_before_approval

### Segmentación

* order_value_segment
* items_segment

---

# Conclusión final

El proceso de Feature Engineering ha permitido transformar múltiples tablas operativas en un dataset analítico único, consistente y preparado para el análisis.

Las validaciones realizadas confirman la coherencia temporal, logística y económica de los datos, así como la correcta construcción de las variables derivadas.

El dataset final constituye una base sólida para desarrollar el análisis exploratorio y estadístico de las siguientes fases del proyecto.
